# Visualize Prescribed Field Generator Data

Generate videos for multiple datasets with moving Gaussian fields.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import torch
import numpy as np
import zarr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm
from pathlib import Path
from IPython.display import Video, display
import matplotlib.animation as animation

from cell_gnn.generators.particle_spring_force_prescribed_field import moving_gaussian_field

In [2]:
# Dataset configs: (name, field_params)
CASES = {
    'case2a': {
        'dataset': 'dicty_case2a_rk4_static_single_gaussian_gnn_siren',
        'title': 'Case 2: Static Single Gaussian (sigma=0.2)',
        'field_params': {
            'center_0': torch.tensor([0.5, 0.5, 0.5]),
            'velocity': torch.tensor([0.0, 0.0, 0.0]),
            'amplitude': 1.0, 'sigma': 0.2, 'mu_chem': 0.02,
            'delta_t': 0.002, 'periodic': True,
        },
    },
    'v4': {
        'dataset': 'dicty_spring_force_rk4_dynamic_field_siren_v4',
        'title': 'v4: Single Moving Gaussian',
        'field_params': {
            'center_0': torch.tensor([0.2, 0.5, 0.5]),
            'velocity': torch.tensor([0.15, 0.0, 0.0]),
            'amplitude': 1.0, 'sigma': 0.15, 'mu_chem': 0.02,
            'delta_t': 0.002, 'periodic': True,
        },
    },
    'v6': {
        'dataset': 'dicty_spring_force_rk4_dynamic_field_siren_v6',
        'title': 'v6: Single Moving Gaussian',
        'field_params': {
            'center_0': torch.tensor([0.2, 0.5, 0.5]),
            'velocity': torch.tensor([0.15, 0.0, 0.0]),
            'amplitude': 1.0, 'sigma': 0.15, 'mu_chem': 0.02,
            'delta_t': 0.002, 'periodic': True,
        },
    },
    'v10': {
        'dataset': 'dicty_spring_force_rk4_dynamic_field_siren_v10',
        'title': 'v10: 8 Moving Gaussians',
        'field_params': {
            'field_type': 'multi',
            'center_0': torch.tensor([
                [0.20, 0.30, 0.50], [0.70, 0.50, 0.50],
                [0.40, 0.80, 0.50], [0.60, 0.20, 0.50],
                [0.30, 0.60, 0.50], [0.50, 0.50, 0.50],
                [0.15, 0.75, 0.50], [0.85, 0.25, 0.50],
            ]),
            'velocity': torch.tensor([
                [ 0.05,  0.00, 0.0], [-0.02,  0.09, 0.0],
                [ 0.00, -0.10, 0.0], [ 0.01,  0.10, 0.0],
                [-0.06, -0.02, 0.0], [ 0.03, -0.07, 0.0],
                [-0.04,  0.05, 0.0], [ 0.05,  0.03, 0.0],
            ]),
            'amplitude': 1.0, 'sigma': 0.05, 'mu_chem': 0.02,
            'delta_t': 0.002, 'periodic': True,
        },
    },
}
print(f'Configured {len(CASES)} cases: {list(CASES.keys())}')

Configured 4 cases: ['case2a', 'v4', 'v6', 'v10']


In [3]:
# Helper: compute field image at z=0.5
grid_res = 100
gx = np.linspace(0, 1, grid_res)
gy = np.linspace(0, 1, grid_res)
GX, GY = np.meshgrid(gx, gy)
grid_xy = np.stack([GX.ravel(), GY.ravel()], axis=1)
grid_3d_base = np.zeros((grid_res * grid_res, 3))
grid_3d_base[:, 0] = grid_xy[:, 0]
grid_3d_base[:, 1] = grid_xy[:, 1]
grid_3d_base[:, 2] = 0.5
grid_t = torch.tensor(grid_3d_base, dtype=torch.float32)

def compute_field_image(t, fp):
    C, _ = moving_gaussian_field(grid_t, t, fp)
    return C.squeeze().numpy().reshape(grid_res, grid_res)

def get_centers(t, fp):
    """Get field center(s) at time t. Returns (S, 3) numpy array."""
    c0 = fp['center_0'].numpy()
    vel = fp['velocity'].numpy()
    if c0.ndim == 1:
        c0 = c0[None, :]
        vel = vel[None, :]
    centers = (c0 + vel * t) % 1.0
    return centers

print('Helpers ready.')

Helpers ready.


In [4]:
save_every = 40
delta_t = 0.002

# Particle display radius (data units) -> scatter s (points²)
# Each panel is 6x6 inches at dpi=100, axis covers ~70% of panel for data range [0,1]
particle_radius = 0.015  # data units
fig_w_per_panel = 6.0
pts_per_data = fig_w_per_panel * 72 * 0.65 / 1.0  # 1 pt = 1/72 inch
radius_pts = particle_radius * pts_per_data
PARTICLE_S = float(np.pi * radius_pts ** 2)
print(f'particle_radius={particle_radius}, scatter s={PARTICLE_S:.1f}')

FIELD_CMAP = 'Oranges'  # field colormap (was Blues)

for key, case in CASES.items():
    dataset = case['dataset']
    title = case['title']
    fp = case['field_params']
    
    data_dir = Path(f'../graphs_data/misc/{dataset}')
    if not (data_dir / 'x_list_0' / 'pos.zarr').exists():
        print(f'SKIP {key}: no data at {data_dir}')
        continue

    print(f'\n{"="*60}')
    print(f'{key}: {dataset}')
    
    pos_z = zarr.open(str(data_dir / 'x_list_0' / 'pos.zarr'), 'r')
    field_z = zarr.open(str(data_dir / 'x_list_0' / 'field.zarr'), 'r')
    n_frames_total = pos_z.shape[0]
    
    frame_indices = np.arange(0, n_frames_total, save_every)
    n_snapshots = len(frame_indices)
    pos_history = np.array([pos_z[i] for i in frame_indices])
    field_history = np.array([field_z[i] for i in frame_indices])
    time_history = frame_indices * delta_t
    
    vmax_field = float(fp['amplitude'])
    out_dir = Path(f'../tmp_training/{dataset}')
    out_dir.mkdir(parents=True, exist_ok=True)
    
    fig = plt.figure(figsize=(18, 6))
    ax1 = fig.add_subplot(131, projection='3d')
    ax2 = fig.add_subplot(132)
    ax3 = fig.add_subplot(133)
    
    def make_update(pos_hist, field_hist, time_hist, fp_, vmax_, title_):
        def update(frame_idx):
            pos = pos_hist[frame_idx]
            t = time_hist[frame_idx]
            fvals = field_hist[frame_idx].squeeze()
            
            ax1.cla()
            ax1.scatter(pos[:, 0], pos[:, 1], pos[:, 2], s=PARTICLE_S, c=fvals, cmap='YlOrRd',
                       vmin=0, vmax=vmax_, alpha=0.7, edgecolors='none')
            ax1.set_xlim(0, 1); ax1.set_ylim(0, 1); ax1.set_zlim(0, 1)
            ax1.set_xlabel('x'); ax1.set_ylabel('y'); ax1.set_zlabel('z')
            ax1.set_title(f'3D  t={t:.3f}')
            
            ax2.cla()
            ax2.scatter(pos[:, 0], pos[:, 1], s=PARTICLE_S, c=fvals, cmap='YlOrRd',
                       vmin=0, vmax=vmax_, alpha=0.7, edgecolors='none')
            ax2.set_xlim(0, 1); ax2.set_ylim(0, 1); ax2.set_aspect('equal')
            ax2.set_xlabel('x'); ax2.set_ylabel('y')
            ax2.set_title('Top-down particles')
            
            ax3.cla()
            field_img = compute_field_image(t, fp_)
            ax3.imshow(field_img, extent=[0, 1, 0, 1], origin='lower',
                       cmap=FIELD_CMAP, vmin=0, vmax=vmax_, alpha=0.9)
            ax3.scatter(pos[:, 0], pos[:, 1], s=PARTICLE_S, c='#1f77b4', alpha=0.7, edgecolors='none')
            centers = get_centers(t, fp_)
            ax3.scatter(centers[:, 0], centers[:, 1], marker='*', c='lime', s=120, zorder=5, edgecolors='k', linewidths=0.5)
            ax3.set_xlim(0, 1); ax3.set_ylim(0, 1); ax3.set_aspect('equal')
            ax3.set_xlabel('x'); ax3.set_ylabel('y')
            ax3.set_title('Field + particles')
            
            fig.suptitle(f'{title_}   t={t:.3f}', fontsize=14)
            fig.tight_layout()
        return update
    
    update_fn = make_update(pos_history, field_history, time_history, fp, vmax_field, title)
    
    print(f'  Rendering {n_snapshots} frames...')
    ani = animation.FuncAnimation(fig, update_fn, frames=n_snapshots, interval=50)
    video_path = str(out_dir / f'{key}_generator.mp4')
    ani.save(video_path, writer='ffmpeg', fps=30, dpi=100)
    plt.close(fig)
    print(f'  Saved: {video_path}')
    
    if key != list(CASES.keys())[-1]:
        fig = plt.figure(figsize=(18, 6))
        ax1 = fig.add_subplot(131, projection='3d')
        ax2 = fig.add_subplot(132)
        ax3 = fig.add_subplot(133)

print('\nDone!')

particle_radius=0.015, scatter s=55.7

case2a: dicty_case2a_rk4_static_single_gaussian_gnn_siren
  Rendering 201 frames...
  Saved: ../tmp_training/dicty_case2a_rk4_static_single_gaussian_gnn_siren/case2a_generator.mp4

v4: dicty_spring_force_rk4_dynamic_field_siren_v4
  Rendering 201 frames...
  Saved: ../tmp_training/dicty_spring_force_rk4_dynamic_field_siren_v4/v4_generator.mp4

v6: dicty_spring_force_rk4_dynamic_field_siren_v6
  Rendering 201 frames...
  Saved: ../tmp_training/dicty_spring_force_rk4_dynamic_field_siren_v6/v6_generator.mp4

v10: dicty_spring_force_rk4_dynamic_field_siren_v10
  Rendering 201 frames...
  Saved: ../tmp_training/dicty_spring_force_rk4_dynamic_field_siren_v10/v10_generator.mp4

Done!


In [6]:
# --- Case 2a: signed grad_c components at z=0.5 ---
# Three panels: ∂c/∂x (signed), ∂c/∂y (signed), ∇c vector field
# Each component shown with viridis on a SYMMETRIC scale [-vmax, +vmax]
# so positive value = field increasing along that axis (yellow),
# negative value = field decreasing (purple).
case2a = CASES['case2a']
case2a_dataset = case2a['dataset']
case2a_fp = case2a['field_params']

grad_res = 60
gx_g = np.linspace(0, 1, grad_res)
gy_g = np.linspace(0, 1, grad_res)
GX_g, GY_g = np.meshgrid(gx_g, gy_g)
grid_3d_grad = np.zeros((grad_res * grad_res, 3))
grid_3d_grad[:, 0] = GX_g.ravel()
grid_3d_grad[:, 1] = GY_g.ravel()
grid_3d_grad[:, 2] = 0.5
grid_t_grad = torch.tensor(grid_3d_grad, dtype=torch.float32)

def compute_grad_c_2d(t, fp):
    _, gradC = moving_gaussian_field(grid_t_grad, t, fp)
    g = gradC.numpy()
    gx2d = g[:, 0].reshape(grad_res, grad_res)
    gy2d = g[:, 1].reshape(grad_res, grad_res)
    return gx2d, gy2d

# Symmetric color scale from t=0
gx0, gy0 = compute_grad_c_2d(0.0, case2a_fp)
vmax_grad = float(max(abs(gx0).max(), abs(gy0).max()) * 1.05)
print(f'case2a |∂c/∂x|, |∂c/∂y| max = {vmax_grad:.4f}')

data_dir2a = Path(f'../graphs_data/misc/{case2a_dataset}')
if not (data_dir2a / 'x_list_0' / 'pos.zarr').exists():
    print(f'SKIP: no data at {data_dir2a}')
else:
    pos_z2a = zarr.open(str(data_dir2a / 'x_list_0' / 'pos.zarr'), 'r')
    n_total2a = pos_z2a.shape[0]
    frame_idxs2a = np.arange(0, n_total2a, save_every)
    n_snap2a = len(frame_idxs2a)
    time_hist2a = frame_idxs2a * delta_t

    out_dir2a = Path(f'../tmp_training/{case2a_dataset}')
    out_dir2a.mkdir(parents=True, exist_ok=True)

    arr_step = 4
    Xa = GX_g[::arr_step, ::arr_step]
    Ya = GY_g[::arr_step, ::arr_step]

    fig = plt.figure(figsize=(18, 6))
    ax_gx = fig.add_subplot(131)
    ax_gy = fig.add_subplot(132)
    ax_vec = fig.add_subplot(133)
    
    # Add fixed colorbars (one for each panel)
    sm_gx = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=-vmax_grad, vmax=vmax_grad))
    cbar_gx = fig.colorbar(sm_gx, ax=ax_gx, fraction=0.046, pad=0.04)
    cbar_gy = fig.colorbar(sm_gx, ax=ax_gy, fraction=0.046, pad=0.04)

    def update_grad(frame_idx):
        t = time_hist2a[frame_idx]
        gxc, gyc = compute_grad_c_2d(t, case2a_fp)

        ax_gx.cla()
        ax_gx.imshow(gxc, extent=[0,1,0,1], origin='lower',
                     cmap='viridis', vmin=-vmax_grad, vmax=vmax_grad)
        ax_gx.set_xlim(0,1); ax_gx.set_ylim(0,1); ax_gx.set_aspect('equal')
        ax_gx.set_xlabel('x'); ax_gx.set_ylabel('y')
        ax_gx.set_title(r'$\partial c/\partial x$  (yellow = +x grad)')

        ax_gy.cla()
        ax_gy.imshow(gyc, extent=[0,1,0,1], origin='lower',
                     cmap='viridis', vmin=-vmax_grad, vmax=vmax_grad)
        ax_gy.set_xlim(0,1); ax_gy.set_ylim(0,1); ax_gy.set_aspect('equal')
        ax_gy.set_xlabel('x'); ax_gy.set_ylabel('y')
        ax_gy.set_title(r'$\partial c/\partial y$  (yellow = +y grad)')

        ax_vec.cla()
        mag = np.sqrt(gxc**2 + gyc**2)
        ax_vec.imshow(mag, extent=[0,1,0,1], origin='lower',
                      cmap='viridis', vmin=0, vmax=vmax_grad, alpha=0.6)
        gxa = gxc[::arr_step, ::arr_step]
        gya = gyc[::arr_step, ::arr_step]
        ax_vec.quiver(Xa, Ya, gxa, gya, color='white',
                      scale=None, width=0.003, pivot='mid')
        ax_vec.set_xlim(0,1); ax_vec.set_ylim(0,1); ax_vec.set_aspect('equal')
        ax_vec.set_xlabel('x'); ax_vec.set_ylabel('y')
        ax_vec.set_title(r'$\nabla c$ vector field')

        fig.suptitle(f'Case 2a: $\\nabla c(\\mathbf{{x}},t)$  t={t:.3f}', fontsize=14)

    print(f'Rendering {n_snap2a} frames for case2a grad_c video...')
    ani = animation.FuncAnimation(fig, update_grad, frames=n_snap2a, interval=50)
    video_path = str(out_dir2a / 'case2a_grad_c.mp4')
    ani.save(video_path, writer='ffmpeg', fps=30, dpi=100)
    plt.close(fig)
    print(f'Saved: {video_path}')

case2a |∂c/∂x|, |∂c/∂y| max = 3.1794
Rendering 201 frames for case2a grad_c video...
Saved: ../tmp_training/dicty_case2a_rk4_static_single_gaussian_gnn_siren/case2a_grad_c.mp4
